# Carbon Mapper Interactive Dashboard — thesis Stage 0

This notebook launches a **Carbon Mapper only** interactive dashboard for querying and exploring high-resolution CH₄/CO₂ plume observations.  
It is intended as the first practical component of the thesis repository: an interactive data-access and exploratory-analysis tool.

**Important:** this notebook intentionally contains **no Sentinel-5P/TROPOMI logic** and no Earth Engine matching. The only external plume source used by the dashboard is the Carbon Mapper API.


## What this notebook does

The dashboard allows the user to:

1. select a country and optional states/provinces;
2. select a date range and gas type, CH₄ and/or CO₂;
3. fetch matching plume records from the Carbon Mapper API;
4. optionally estimate plume area from available Carbon Mapper raster links;
5. summarize plume counts, plume areas, and available emission fields by administrative region;
6. visualize results using bar charts, histograms, choropleth maps, plume points, and optional heatmaps;
7. export or download the raw Carbon Mapper CSV queried by the app.

The implementation lives in the companion Python module:

```text
cm_app_carbonmapper_only.py
```

Keep this `.ipynb` file and the `.py` file in the same repository folder.


## Repository structure recommended for GitHub

A simple structure is enough for this first thesis component:

```text
thesis/
├── 01_carbon_mapper_dashboard/
│   ├── launch_carbonmapper_dashboard_only.ipynb
│   └── cm_app_carbonmapper_only.py
├── README.md
└── .gitignore
```

Do **not** commit API tokens or generated data outputs. The dashboard writes outputs to:

```text
cm_app_outputs/
```

Add this folder to `.gitignore` if you do not want raw downloaded CSVs in GitHub.


In [ ]:
# Install the Python packages required by the Carbon Mapper dashboard.
# In Colab this cell is usually needed once per runtime.
# In a local environment, you can instead install these packages in your virtual environment.

!pip -q install geopandas rasterio folium ipywidgets shapely pandas numpy requests matplotlib

# Enable ipywidgets rendering in Google Colab.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    # This notebook can also run outside Colab, where google.colab is unavailable.
    pass


## Locate the companion Python module

The next cell searches for `cm_app_carbonmapper_only.py`, adds its folder to `sys.path`, and then imports the dashboard class.

If the module is not found, make sure the `.py` file is uploaded to the same Colab runtime or stored in your mounted Google Drive folder.


In [ ]:
import os
import sys
from pathlib import Path

MODULE_FILE = "cm_app_carbonmapper_only.py"

# Mount Google Drive if running in Colab. This is useful when the repository is stored in Drive.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

candidate_paths = []

# 1) Current working directory
candidate_paths.append(Path.cwd() / MODULE_FILE)

# 2) Common Colab upload location
candidate_paths.append(Path("/content") / MODULE_FILE)

# 3) Common thesis folder location in Drive
candidate_paths.append(Path("/content/drive/MyDrive/thesis") / MODULE_FILE)

# 4) Search MyDrive recursively if available
mydrive = Path("/content/drive/MyDrive")
if mydrive.exists():
    candidate_paths.extend(mydrive.glob(f"**/{MODULE_FILE}"))

module_path = next((p for p in candidate_paths if p.exists()), None)

if module_path is None:
    raise FileNotFoundError(
        f"Could not find {MODULE_FILE}. Upload it to /content, place it beside this notebook, "
        "or update the search path in this cell."
    )

module_dir = str(module_path.parent)
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

print(f"Using dashboard module: {module_path}")

from cm_app_carbonmapper_only import CMApp


## Provide the Carbon Mapper API token safely

The dashboard needs a Carbon Mapper API token. For GitHub safety, the token is **not stored in this notebook**.

When you run the next cell, paste your token into the hidden input prompt. The token will only be stored in the current runtime environment variable `CM_TOKEN`.

Do not write the token directly into a notebook cell before uploading to GitHub.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("CM_TOKEN"):
    os.environ["CM_TOKEN"] = getpass("Paste your Carbon Mapper API token: ").strip()

if not os.environ.get("CM_TOKEN"):
    raise ValueError("CM_TOKEN is empty. Please provide a valid Carbon Mapper API token.")

print("Carbon Mapper token is set for this runtime session.")


## Launch the dashboard

Run the next cell to display the dashboard.

Suggested first test:

- Country: `United States of America`
- Date range: a short interval first, for example one month
- Gas: `CH4`
- Max plumes: keep it small for testing, for example `500`
- Leave states/provinces empty to query all available regions in the selected country

After the first test works, increase the date range or select specific states/provinces.


In [ ]:
app = CMApp(cm_token=os.environ["CM_TOKEN"])
app.display()


## Optional: export state/province statistics after running the dashboard

After clicking **Fetch & Analyze** in the dashboard, you can export summary statistics with the helper method below.

Change `gas="CH4"` to `gas="CO2"` if needed.


In [ ]:
# Run this only after the dashboard has loaded plume data.
# app.export_state_stats(path="carbon_mapper_state_stats_ch4.csv", gas="CH4")


## Optional: save the interactive map as HTML

After running **Fetch & Analyze**, the current map can be saved as an HTML file.

The saved file can be opened in a browser or included as a supplementary interactive artifact.


In [ ]:
# Run this only after the dashboard has loaded plume data.
# app.save_map(path="carbon_mapper_ch4_map.html", gas="CH4", metric="plume_count")


## Notes for thesis documentation

This dashboard should be described as an **interactive data-access and exploratory-analysis interface** for Carbon Mapper plume observations.

A scientifically safe description is:

> The dashboard queries Carbon Mapper plume metadata for a selected country, administrative region, date interval, and gas type. It then summarizes and visualizes plume occurrence and available plume attributes by administrative unit. Optional plume-area estimation is based on available Carbon Mapper raster products and is used for exploratory comparison rather than formal emission-rate validation.

Avoid describing this tool as a validation system. It is a query, visualization, and exploratory analysis dashboard.


## GitHub safety checklist

Before uploading to GitHub:

- confirm there is no API token in this notebook;
- confirm there is no API token in `cm_app_carbonmapper_only.py`;
- remove notebook outputs if they contain private paths or large tables;
- add generated outputs to `.gitignore`, especially `cm_app_outputs/`;
- include a short README explaining that users must provide their own Carbon Mapper API token.

Suggested `.gitignore` entries:

```text
cm_app_outputs/
*.html
*.csv
__pycache__/
.ipynb_checkpoints/
```
